> Many sprints later, facing a €1.2 billion fine, the engineers would remember that afternoon when someone decided the right to erasure was a product concern, not a property of the cache.

# vLLM Prefix Caching — Production Architecture for Fashion Retail

This notebook demonstrates **prefix caching** in vLLM applied to a real-world fit-feedback moderation workload: classifying free-text comments at scale under Digital Services Act (DSA) content moderation obligations and General Data Protection Regulation (GDPR) deletion requirements.

**Runtime**: T4 Graphics Processing Unit (GPU) — free Colab tier is sufficient  
**Model**: `Qwen/Qwen2.5-7B-Instruct-AWQ` (~4 GB, fits on T4)

## 1. The Mechanism

Every LLM request arrives twice: once as computation, once as memory. vLLM hashes the context in blocks of sixteen tokens, chaining each to the previous, and caches the result. A matching hash on the next request means free prefill. The cache remembers. GDPR requires forgetting. These two facts produce a liability invisible until it surfaces in an audit, a complaint, or a fine.

| Phase | What happens | Cost |
|-------|-------------|------|
| **Prefill** | Compute Key-Value (KV) pairs for every input token | O(n²) in context length — **slow** |
| **Decode** | Generate output tokens one by one | O(n) per token — fast |

```
block 1: hash([policy tokens])                → h1  ← reused across ALL requests
block 2: hash([product review tokens] + h1)   → h2  ← reused per SKU
block 3: hash([comment + audit token] + h2)   → h3  ← always computed fresh
```

Ordering layers from most-stable to least-stable lets cache hits cascade down the hierarchy.

## 2. The Use Case

Twenty-two million, three hundred thousand times per year, a person submits a sentence about how a garment fit their body. Each carries three simultaneous obligations: moderated under DSA Article 17, deletable under GDPR Article 17, every decision traceable by session for audit. At this volume, compliance is not a budget line. It is a direct liability compounding with each event, each erasure, each audit finding.

```
┌─────────────────────────────────────────────────────────┐
│ Layer 1: DSA content moderation policy                  │
│          ~160 tokens · invariant across all 22M requests│
│          → pre-warm ONCE at server startup              │
├─────────────────────────────────────────────────────────┤
│ Layer 2: Product fit reviews for a specific SKU         │
│          ~120 tokens · stable until a GDPR erasure      │
│          → pre-warm PER SKU at catalog load             │
├─────────────────────────────────────────────────────────┤
│ Layer 3: DSA audit token + the comment being classified │
│          ~60 tokens · unique per request                │
│          → always computed fresh                        │
└─────────────────────────────────────────────────────────┘
```

**Layer ordering is the design constraint.** DSA policy first — most stable, highest reuse. Reviews second — stable until erasure. Audit token and comment last — dynamic, unique per request. Reversing any layer defeats the cache for every layer below it.

In [ ]:
import sys, subprocess, os

try:
    import vllm
    print(f">>> vllm {vllm.__version__} — ready")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm<0.18.0"], check=True)
    print(">>> vllm installed — restarting, then Run all again")
    os.kill(os.getpid(), 9)

In [ ]:
MODEL_HF      = "Qwen/Qwen2.5-7B-Instruct-AWQ"
GPU_MEM_UTIL  = 0.85
MAX_MODEL_LEN = 4096

# Platform scale constants — used in Experiments 4 and 5
ANNUAL_MODERATION_EVENTS = 22_300_000   # fit comments per year
CATALOG_SIZE             = 100_000      # distinct SKUs
AVG_REVIEWS_PER_SKU      = 20          # reviews in the product prefix
MONTHLY_ERASURE_RATE     = 0.001       # 0.1% of review authors request deletion/month

In [ ]:
import time
import textwrap
from vllm import LLM, SamplingParams

In [ ]:
llm = LLM(
    model=MODEL_HF,
    enable_prefix_caching=True,
    enforce_eager=True,
    gpu_memory_utilization=GPU_MEM_UTIL,
    max_model_len=MAX_MODEL_LEN,
)

sampling = SamplingParams(temperature=0, max_tokens=20)

## 3. Experiment 1 — Two-Layer Prefix: Policy + Product Reviews

The DSA policy is invariant across all twenty-two million requests — computed once, reused indefinitely. Product reviews are stable per SKU until a reviewer exercises the right to erasure. Until that moment, every warm request is a unit of liability satisfied cheaply. The cold request that follows an erasure is the infrastructure cost of having satisfied the legal one.

In [ ]:
# Layer 1: DSA content moderation policy — identical across all 22M requests
POLICY = """\
Content Moderation Policy v2.1 — Fit Feedback Classification

Classify user-submitted fit feedback into exactly one category:
- SAFE       : comment discusses product fit, sizing, comfort, or return reason only
- FLAG:PII   : comment contains identifiable personal information
- FLAG:HATE  : comment contains discriminatory or abusive language
- FLAG:SPAM  : comment is promotional or unrelated to product fit
- FLAG:REVIEW: ambiguous — escalate to human moderator

Rules:
1. Classify based on content only, not sentiment about the product.
2. Sizing opinions ("runs small", "too narrow") are always SAFE.
3. A name mentioned in a gift context is not FLAG:PII.
4. Respond with ONLY the classification label on a single line.
"""

# Layer 2: Product reviews for SKU-1042 — stable until a GDPR Article 17 erasure request
REVIEWS = """\
Fit feedback for Structured Blazer SKU-1042:
- "fits my waist but I can't button it after lunch"
- "bought both M and L — the M looks better but L is more comfortable"
- "first time buying this brand, not sure about European sizing"
- "I'm between sizes and this is a structured blazer"
"""

# Layer 3: Comments to classify — unique per request
COMMENTS = [
    "Ordered M, way too tight across the shoulders. Returning.",
    "True to size for me, keeping it.",
    "john.smith@email.com — please process my return",
]

def make_prompt(comment):
    return (
        f"<|im_start|>system\n{POLICY}\n{REVIEWS}<|im_end|>\n"
        f"<|im_start|>user\nClassify: {comment}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

tokenizer = llm.get_tokenizer()
shared_tokens = len(tokenizer.encode(f"<|im_start|>system\n{POLICY}\n{REVIEWS}<|im_end|>"))
print(f"Shared prefix (policy + reviews): {shared_tokens} tokens")
print()

for i, comment in enumerate(COMMENTS):
    prompt = make_prompt(comment)
    t0 = time.perf_counter()
    output = llm.generate([prompt], sampling)
    elapsed = time.perf_counter() - t0
    label = "COLD (full prefill)" if i == 0 else "WARM (cache hit)   "
    result = output[0].outputs[0].text.strip()
    print(f"[{label}] {elapsed:.2f}s → {result}")
    print(f"  Comment: {textwrap.shorten(comment, 80)}")
    print()

## 4. Experiment 2 — GDPR Article 17 Erasure Invalidates the Cache

One reviewer. One erasure request. The review disappears — the obligation is discharged. But its removal changes the token sequence, changes the block hash, and invalidates every block chained below it. The cached prefix empties. The next request pays full prefill cost, as if six months of caching never happened. This is correct behavior. The remediation cost — GPU time rebuilding what erasure erased — is a scheduled liability, predictable as the erasure rate that drives it: zero point one percent monthly across one hundred thousand SKUs is one thousand two hundred prefix invalidations per year. Each a cascade. Each a line item that regulators reviewing audit logs will name — whether or not the engineering team has.

In [ ]:
RAW_REVIEWS = [
    "fits my waist but I can't button it after lunch",
    "bought both M and L — the M looks better but L is more comfortable",
    "first time buying this brand, not sure about European sizing",
    "I'm between sizes and this is a structured blazer",
]

def make_prompt_with_reviews(comment, reviews):
    review_block = "Fit feedback for Structured Blazer SKU-1042:\n" + "\n".join(f'- "{r}"' for r in reviews)
    return (
        f"<|im_start|>system\n{POLICY}\n{review_block}\n<|im_end|>\n"
        f"<|im_start|>user\nClassify: {COMMENTS[0]}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

# Warm the cache with the full review set
llm.generate([make_prompt_with_reviews(COMMENTS[0], RAW_REVIEWS)], sampling)

# Full review set — cache hit
t0 = time.perf_counter()
llm.generate([make_prompt_with_reviews(COMMENTS[0], RAW_REVIEWS)], sampling)
t_cached = time.perf_counter() - t0

# GDPR erasure: reviewer index 1 requests deletion
reviews_after_erasure = [r for i, r in enumerate(RAW_REVIEWS) if i != 1]
t0 = time.perf_counter()
llm.generate([make_prompt_with_reviews(COMMENTS[0], reviews_after_erasure)], sampling)
t_erased = time.perf_counter() - t0

print(f"Full review set (cache hit):     {t_cached:.2f}s")
print(f"After GDPR erasure (cache miss): {t_erased:.2f}s")
print(f"Erasure penalty:                 {t_erased/t_cached:.1f}x slower")
print()

# Scale: monthly erasure impact across the catalog
monthly_erasures = int(CATALOG_SIZE * AVG_REVIEWS_PER_SKU * MONTHLY_ERASURE_RATE)
monthly_prefill_cost_s = monthly_erasures * t_erased

print(f"At platform scale ({CATALOG_SIZE:,} SKUs, {AVG_REVIEWS_PER_SKU} reviews/SKU):")
print(f"  Monthly erasure requests:      ~{monthly_erasures:,}")
print(f"  Cache invalidations/month:     ~{monthly_erasures:,} SKU prefixes")
print(f"  Re-warm GPU cost (sequential): ~{monthly_prefill_cost_s/60:.0f} minutes")
print()
print("Mitigation: schedule re-warm of invalidated SKUs during off-peak hours.")

## 5. Experiment 3 — Audit Token Placement as a DSA Compliance Constraint

DSA Article 17 is non-negotiable. The audit token is mandatory; its absence is a compliance failure; its wrong position is an architectural one with a measurable annual cost. The cache and the regulation agree: stable shared prefix, dynamic per-request content at the end. The token belongs there. Placed correctly, it satisfies both simultaneously — no trade-off, no remediation, no exposure. Placed before the policy, it poisons every block below it. The annual GPU-hour cost is computed below. The reputational cost of a missed audit trail during a regulatory investigation is not.

In [ ]:
import uuid

def make_prompt_audit_prefix(comment):
    """WRONG: audit token before policy — cache miss from block 1."""
    audit_id = f"[audit:{uuid.uuid4().hex[:12]}]"
    return (
        f"<|im_start|>system\n{audit_id}\n{POLICY}\n{REVIEWS}<|im_end|>\n"
        f"<|im_start|>user\nClassify: {comment}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

def make_prompt_audit_suffix(comment):
    """CORRECT: audit token in user turn — cache hit on shared prefix."""
    audit_id = f"[audit:{uuid.uuid4().hex[:12]}]"
    return (
        f"<|im_start|>system\n{POLICY}\n{REVIEWS}<|im_end|>\n"
        f"<|im_start|>user\n{audit_id} Classify: {comment}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

comment = COMMENTS[0]
llm.generate([make_prompt(comment)], sampling)  # warm cache

t0 = time.perf_counter()
llm.generate([make_prompt_audit_prefix(comment)], sampling)
t_bad = time.perf_counter() - t0

t0 = time.perf_counter()
llm.generate([make_prompt_audit_suffix(comment)], sampling)
t_good = time.perf_counter() - t0

print(f"Audit token BEFORE shared prefix (cache miss): {t_bad:.2f}s")
print(f"Audit token AFTER  shared prefix (cache hit):  {t_good:.2f}s")
print(f"Speedup from compliant placement: {t_bad/t_good:.1f}x")
print()

# Annual compute cost of the wrong placement
extra_cost_per_request_s = t_bad - t_good
annual_wasted_gpu_hours = (extra_cost_per_request_s * ANNUAL_MODERATION_EVENTS) / 3600
print(f"Annual GPU time wasted by wrong placement: {annual_wasted_gpu_hours:,.0f} GPU-hours")
print("DSA audit logging requirement is satisfied by both placements.")
print("Only the token position differs — with a consequence at 22M requests/year.")

## 6. Experiment 4 — Catalog-Scale Cache Planning: 100K SKUs

One hundred thousand SKUs, each a distinct cached prefix, each an eviction question answered at runtime. Eviction is a procurement decision expressed as a cache miss rate. The Zipf distribution concentrates eighty percent of traffic on twenty percent of SKUs — a hot tier sized to that fraction eliminates most eviction exposure. The minimum VRAM to hold the full catalog is computed below. It fits the production GPU budget — but only because someone ran this calculation before the hardware order, not after the remediation cost appeared on the balance sheet.

In [ ]:
# Qwen2.5-7B architecture constants
LAYERS   = 28
KV_HEADS = 8
HEAD_DIM = 128

values_per_token      = LAYERS * 2 * KV_HEADS * HEAD_DIM
int4_bytes_per_token  = values_per_token * 0.5

# Per-SKU prefix: policy layer + review layer
policy_tokens  = 160
reviews_tokens = 120
sku_prefix_tokens = policy_tokens + reviews_tokens

kv_per_sku_bytes = int4_bytes_per_token * sku_prefix_tokens

# T4: ~11 GB available after model weights
kv_vram_t4_bytes = 11 * 1e9
hot_skus_t4 = int(kv_vram_t4_bytes / kv_per_sku_bytes)

# Production GPU: 96 GB GDDR7, model ~4 GB, leaving 92 GB
kv_vram_prod_bytes = 92 * 1e9
hot_skus_prod = int(kv_vram_prod_bytes / kv_per_sku_bytes)

# Eviction rate and daily cold-prefill cost (T4)
# Zipf traffic distribution: top 20% SKUs → 80% of requests
hot_fraction_t4  = min(hot_skus_t4 / CATALOG_SIZE, 1.0)
eviction_rate_t4 = 1.0 - hot_fraction_t4
daily_requests   = ANNUAL_MODERATION_EVENTS / 365
cold_requests    = daily_requests * eviction_rate_t4 * 0.20   # cold SKUs get ~20% traffic
cold_gpu_min     = cold_requests * t_erased / 60

print(f"Per-SKU prefix: {sku_prefix_tokens} tokens")
print(f"  INT4 Key-Value (KV) size: {kv_per_sku_bytes/1024:.0f} KB per SKU")
print()
print(f"T4 (11 GB KV budget):")
print(f"  Hot-tier capacity:  {hot_skus_t4:,} of {CATALOG_SIZE:,} SKUs ({hot_fraction_t4*100:.0f}%)")
print(f"  Daily cold-prefill: {cold_gpu_min:.0f} GPU-minutes")
print()
print(f"Production GPU (92 GB KV budget, INT4):")
print(f"  Hot-tier capacity:  {hot_skus_prod:,} SKUs")
if hot_skus_prod >= CATALOG_SIZE:
    print(f"  Entire {CATALOG_SIZE:,}-SKU catalog fits — zero LRU eviction under normal load.")
else:
    print(f"  {CATALOG_SIZE - hot_skus_prod:,} SKUs spill to warm/cold tier.")

## 7. KV Cache Quantization: FP16 vs INT4 at Catalog Scale

At one hundred thousand SKUs, FP16 versus INT4 is not a quality decision. It is a capacity decision — the margin between a cache that holds the full catalog and one that evicts continuously, compounding remediation costs on top of erasure costs on top of audit overhead. INT4 changes one number: how many SKU prefixes survive a working day in hot-tier memory. At production GPU size, that number is enough. Whether the procurement team knew this calculation existed is a different question.

In [ ]:
fp16_bytes_per_token = values_per_token * 2

print(f"KV size per token:")
print(f"  FP16: {fp16_bytes_per_token/1024:.1f} KB/token")
print(f"  INT4: {int4_bytes_per_token/1024:.1f} KB/token")
print()

# SKU hot-tier capacity comparison
skus_fp16_t4   = int(kv_vram_t4_bytes   / (fp16_bytes_per_token * sku_prefix_tokens))
skus_int4_t4   = int(kv_vram_t4_bytes   / (int4_bytes_per_token * sku_prefix_tokens))
skus_fp16_prod = int(kv_vram_prod_bytes  / (fp16_bytes_per_token * sku_prefix_tokens))
skus_int4_prod = int(kv_vram_prod_bytes  / (int4_bytes_per_token * sku_prefix_tokens))

print(f"SKU prefixes ({sku_prefix_tokens} tokens each) in hot tier:")
print(f"  T4  11 GB  FP16: {skus_fp16_t4:,} SKUs")
print(f"  T4  11 GB  INT4: {skus_int4_t4:,} SKUs  (4× more)")
print(f"  Prod 92 GB FP16: {skus_fp16_prod:,} SKUs")
print(f"  Prod 92 GB INT4: {skus_int4_prod:,} SKUs  (4× more)")
print()

# Breakeven: minimum VRAM to hold the full 100K-SKU catalog without eviction
min_vram_fp16_gb = (fp16_bytes_per_token * sku_prefix_tokens * CATALOG_SIZE) / 1e9
min_vram_int4_gb = (int4_bytes_per_token * sku_prefix_tokens * CATALOG_SIZE) / 1e9

print(f"Minimum VRAM to hold full {CATALOG_SIZE:,}-SKU catalog (no eviction):")
print(f"  FP16: {min_vram_fp16_gb:.1f} GB")
print(f"  INT4: {min_vram_int4_gb:.1f} GB  ← fits in production GPU with room to spare")

## Summary

| Prefix layer | Stability | Cache strategy |
|---|---|---|
| DSA policy | Never changes | Warm once at startup; reused for all 22M requests |
| Product reviews (per SKU) | Stable until GDPR Article 17 erasure | Warm per SKU at catalog load; re-warm after deletion |
| Audit token + comment | Unique per request | Always at the end of the prompt; never busts upper layers |

| Mechanism | Production consequence |
|---|---|
| Two-layer prefix hierarchy | Policy computed once; review layer reused per SKU |
| Layer ordering | DSA policy first, reviews second, audit token last — non-negotiable |
| GDPR erasure cost | Each deletion invalidates one SKU prefix; schedule re-warm off-peak |
| INT4 KV quantization | 4× more SKU prefixes in Video Random Access Memory (VRAM) — reduces cold-prefill frequency |
| Production GPU (92 GB KV) | Entire 100K-SKU catalog fits in hot tier — zero Least Recently Used (LRU) eviction under normal load |

Compliance and efficiency are not in conflict. The audit token belongs at the end — the law and the cache agree. GDPR erasure has a measurable remediation cost: name it, schedule it, budget it before it appears on the infrastructure invoice. The only question is sequence: does your organization discover these numbers in a planning session, or in an enforcement order, a suspension notice, and a fine calibrated against annual turnover?